# Comparaison des méthodes

Ce notebook rassemble **plusieurs dossiers de run** (même dataset, même architecture, même K) et compare :
ta méthode (pour chaque α), l'ensemble indépendant (α = 0), la DML de Zhang et al. (additive ou multiplicative)
et le grand agent de `collab_vs_big`.

Toutes les statistiques sont des **médianes [quantile 5 % ; quantile 95 %] sur les graines**. La comparaison
« graine par graine » (différence appariée) est plus fiable que la comparaison des médianes : deux méthodes
lancées avec la même graine voient les mêmes initialisations et les mêmes données.

Les figures et le tableau sont enregistrés dans `runs/comparaison_<arch>_<dataset>_x<K>/`.

In [ ]:
import glob
import json
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from IPython.display import display

# --- À régler -------------------------------------------------------------
RUNS_ROOT = "../runs"
DATASET = "fashion_mnist"
ARCH = "resnet20"
K = 3
RUN_DIRS = None      # None = tous les runs alpha_sweep / collab_vs_big de ARCH, DATASET, K (avec ou sans tag)
                     # sinon une liste, ex. ["../runs/alpha_sweep_resnet20_cifar10_x3", "../runs/alpha_sweep_resnet20_cifar10_x3_dml"]
REFERENCE = None     # méthode de référence pour les différences appariées (None : grand agent, sinon ensemble indépendant)
# ---------------------------------------------------------------------------

if RUN_DIRS is None:
    RUN_DIRS = sorted(d for exp in ("alpha_sweep", "collab_vs_big")
                      for d in glob.glob(os.path.join(RUNS_ROOT, f"{exp}_{ARCH}_{DATASET}_x{K}*"))
                      if os.path.exists(os.path.join(d, "results.json")))
if not RUN_DIRS:
    raise FileNotFoundError(f"Aucun run terminé pour {ARCH} / {DATASET} / K = {K} dans {os.path.abspath(RUNS_ROOT)}")
OUT_DIR = os.path.join(RUNS_ROOT, f"comparaison_{ARCH}_{DATASET}_x{K}")
os.makedirs(OUT_DIR, exist_ok=True)
for d in RUN_DIRS:
    print(os.path.basename(d))

## Lecture des runs

Une ligne par (méthode, graine). Le nom de la méthode vient de la config du run : loss, forme, α.
Avec α = 0, il n'y a pas d'interaction : ce sont des agents **indépendants** (L_k² en multiplicatif, CE en additif).

In [ ]:
FORM_SHORT = {"multiplicative": "mult.", "additive": "add."}
FAMILY_OF_LOSS = {"dml": "DML"}

def method_name(cfg, alpha):
    form = FORM_SHORT.get(cfg.get("loss_form"), cfg.get("loss_form"))
    if alpha == 0:
        return f"Indépendants ({'CE' if cfg.get('loss_form') == 'additive' else 'L²'})", "Indépendants"
    loss = cfg.get("loss_type")
    family = FAMILY_OF_LOSS.get(loss, "Ta méthode")
    label = f"DML {form}" if loss == "dml" else f"{loss} {form}"
    return f"{label} α={alpha:g}", family

def last(history, key):
    values = history.get(key) or []
    return values if isinstance(values, list) else []

rows, curves = [], []
for d in RUN_DIRS:
    cfg = json.load(open(os.path.join(d, "config.json")))
    res = json.load(open(os.path.join(d, "results.json")))
    tag = cfg.get("tag")
    if cfg.get("experiment") == "alpha_sweep" or "alphas" in cfg:
        for r in res["runs"]:
            name, family = method_name(cfg, float(r["alpha"]))
            rows.append(dict(method=name, family=family, tag=tag, run_dir=os.path.basename(d), seed=r["seed"],
                             consensus=r["accuracy_consensus"], ensemble=r["accuracy_mean_ensemble"],
                             agent_median=r["accuracy_agent_median"], agent_best=r["accuracy_agent_best"]))
            curves.append(dict(method=name, seed=r["seed"], tag=tag, run_dir=os.path.basename(d),
                               val=last(r["history"], "val_accuracy")))
    else:   # collab_vs_big : petits agents + grand agent
        sizes = res.get("model_sizes", {})
        for r in res["runs"]:
            s, b = r["small_agents"], r["big_agent"]
            name, family = method_name(cfg, float(cfg.get("alpha", 1.0)))
            rows.append(dict(method=name + " (collab_vs_big)", family=family, tag=tag, run_dir=os.path.basename(d),
                             seed=r["seed"], consensus=s["accuracy_consensus"], ensemble=s["accuracy_mean_ensemble"],
                             agent_median=s["accuracy_agent_median"], agent_best=s["accuracy_agent_best"]))
            curves.append(dict(method=name + " (collab_vs_big)", seed=r["seed"], tag=tag, run_dir=os.path.basename(d),
                               val=last(s["history"], "val_accuracy")))
            big = f"Grand agent (width {sizes.get('big_width', '?')})"
            rows.append(dict(method=big, family="Grand agent", tag=tag, run_dir=os.path.basename(d), seed=r["seed"],
                             consensus=b["accuracy"], ensemble=b["accuracy"], agent_median=b["accuracy"],
                             agent_best=b["accuracy"]))
            curves.append(dict(method=big, seed=r["seed"], tag=tag, run_dir=os.path.basename(d),
                               val=last(b["history"], "val_accuracy")))

df = pd.DataFrame(rows)
# deux runs différents qui donnent le même nom (ex. même loss, lr différents) : on ajoute le tag
dup = df.groupby("method")["run_dir"].nunique()
clash = set(dup[dup > 1].index)
df["method"] = [f"{m} [{t or 'sans tag'}]" if m in clash else m for m, t in zip(df["method"], df["tag"])]
curves_df = pd.DataFrame(curves)
if len(curves_df):
    curves_df["method"] = [f"{m} [{t or 'sans tag'}]" if m in clash else m
                           for m, t in zip(curves_df["method"], curves_df["tag"])]
print(f"{df['method'].nunique()} méthodes, graines : {sorted(int(s) for s in df['seed'].unique())}")

## Tableau récapitulatif

Médiane [q05 ; q95] sur les graines, triée par accuracy du consensus.
- **consensus** : prédiction du système (stratégie de la config ; vote majoritaire pour α = 0) ;
- **ensemble** : moyenne simple des probabilités des agents ;
- **agent médian / meilleur agent** : un agent seul, comme Zhang et al. évaluent la DML.

In [ ]:
METRICS = {"consensus": "Consensus", "ensemble": "Moyenne simple", "agent_median": "Agent médian", "agent_best": "Meilleur agent"}

def fmt(values):
    v = np.asarray(values, dtype=float)
    return f"{np.median(v):.4f} [{np.quantile(v, 0.05):.4f} ; {np.quantile(v, 0.95):.4f}]"

order = df.groupby("method")["consensus"].median().sort_values(ascending=False).index
table = pd.DataFrame({label: df.groupby("method")[key].apply(fmt) for key, label in METRICS.items()}).loc[order]
table.insert(0, "Famille", df.groupby("method")["family"].first().loc[order])
table["Graines"] = df.groupby("method")["seed"].nunique().loc[order]
table.to_csv(os.path.join(OUT_DIR, "comparaison.csv"))
display(table)

## Accuracy par méthode

Point plein : consensus. Point creux : agent médian. Barre : intervalle [q05 ; q95] sur les graines.

In [ ]:
# Couleur par famille (palette catégorielle validée ; gris pour la référence « grand agent »)
FAMILY_COLORS = {"Ta méthode": "#2a78d6", "DML": "#eb6834", "Indépendants": "#1baf7a", "Grand agent": "#8a8984"}
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e6e5e0"

def style(ax):
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(MUTED)
    ax.tick_params(colors=MUTED, labelcolor=INK)
    ax.grid(axis="x", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)

methods = list(order)[::-1]                    # meilleure méthode en haut
fig, ax = plt.subplots(figsize=(8, 0.45 * len(methods) + 1.4))
for i, m in enumerate(methods):
    sub = df[df["method"] == m]
    color = FAMILY_COLORS.get(sub["family"].iloc[0], MUTED)
    for key, offset, filled in (("consensus", 0.12, True), ("agent_median", -0.12, False)):
        v = sub[key].to_numpy()
        med, lo, hi = np.median(v), np.quantile(v, 0.05), np.quantile(v, 0.95)
        ax.plot([lo, hi], [i + offset] * 2, color=color, linewidth=2, solid_capstyle="round")
        ax.plot(med, i + offset, "o", markersize=8, markeredgewidth=2, markeredgecolor=color,
                markerfacecolor=color if filled else "white", zorder=3)
ax.set_yticks(range(len(methods)), methods)
ax.set_xlabel("accuracy test", color=INK)
style(ax)
handles = [plt.Line2D([], [], marker="o", linestyle="", markersize=8, markeredgewidth=2, markeredgecolor=MUTED,
                      markerfacecolor=f, label=l) for f, l in ((MUTED, "consensus"), ("white", "agent médian"))]
handles += [plt.Line2D([], [], color=c, linewidth=6, label=f) for f, c in FAMILY_COLORS.items()
            if f in set(df["family"])]
ax.legend(handles=handles, loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)
ax.set_title(f"{ARCH} x{K} sur {DATASET} : médiane [q05 ; q95] sur les graines", color=INK, loc="left")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "accuracy_methodes.png"), dpi=200, bbox_inches="tight")
plt.show()

## Différence appariée avec la référence

Pour chaque graine : accuracy du consensus de la méthode − accuracy de la référence **avec la même graine**.
Une méthode dont tous les points sont à droite de 0 bat la référence sur chaque graine.
Référence par défaut : le grand agent s'il existe, sinon l'ensemble indépendant.

In [ ]:
ref = REFERENCE
if ref is None:
    for family in ("Grand agent", "Indépendants"):
        names = df.loc[df["family"] == family, "method"].unique()
        if len(names):
            ref = sorted(names)[0]
            break
ref_acc = df[df["method"] == ref].set_index("seed")["consensus"]
others = [m for m in methods if m != ref]
fig, ax = plt.subplots(figsize=(8, 0.45 * len(others) + 1.4))
summary = {}
for i, m in enumerate(others):
    sub = df[df["method"] == m].set_index("seed")
    common = sub.index.intersection(ref_acc.index)
    if not len(common):
        continue
    diff = (sub.loc[common, "consensus"] - ref_acc.loc[common]).to_numpy() * 100
    color = FAMILY_COLORS.get(sub["family"].iloc[0], MUTED)
    jitter = np.linspace(-0.15, 0.15, len(diff)) if len(diff) > 1 else [0.0]
    ax.plot(diff, i + np.asarray(jitter), "o", markersize=8, color=color, markeredgecolor="white", markeredgewidth=2)
    ax.plot(np.median(diff), i, "|", markersize=18, markeredgewidth=2, color=INK)
    summary[m] = (np.median(diff), int((diff > 0).sum()), len(diff))
ax.axvline(0, color=MUTED, linewidth=1)
ax.set_yticks(range(len(others)), others)
ax.set_xlabel(f"écart d'accuracy avec « {ref} » (points de %)", color=INK)
style(ax)
ax.set_title("Différence appariée par graine (trait noir : médiane)", color=INK, loc="left")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "difference_appariee.png"), dpi=200, bbox_inches="tight")
plt.show()
for m, (med, wins, n) in sorted(summary.items(), key=lambda kv: -kv[1][0]):
    print(f"{m:45s} {med:+6.2f} pts (médiane) | meilleure sur {wins}/{n} graines")

## Courbes d'apprentissage

Accuracy de validation par époque (consensus ; le modèle seul pour le grand agent), médiane sur les graines,
zone = [q05 ; q95]. Une couleur par famille ; quand une famille a plusieurs méthodes, les traits se distinguent
par le style (plein, tirets, pointillés).

In [ ]:
if len(curves_df) and curves_df["val"].map(len).max() > 0:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    dashes = ["-", "--", ":", "-."]
    seen = {}
    fam = df.groupby("method")["family"].first()
    for m in order:
        sub = curves_df[curves_df["method"] == m]
        if not len(sub):
            continue
        n = min(len(v) for v in sub["val"])
        if n == 0:
            continue
        arr = np.array([v[:n] for v in sub["val"]])
        x = np.arange(1, n + 1)
        family = fam.get(m, "")
        color = FAMILY_COLORS.get(family, MUTED)
        seen[family] = seen.get(family, -1) + 1
        ls = dashes[seen[family] % len(dashes)]
        ax.fill_between(x, np.quantile(arr, 0.05, axis=0), np.quantile(arr, 0.95, axis=0), color=color, alpha=0.12, linewidth=0)
        ax.plot(x, np.median(arr, axis=0), color=color, linestyle=ls, linewidth=2, label=m)
    ax.set_xlabel("époque", color=INK)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_ylabel("accuracy validation", color=INK)
    style(ax)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)
    ax.set_title("Courbes d'apprentissage (médiane sur les graines)", color=INK, loc="left")
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "courbes_validation.png"), dpi=200, bbox_inches="tight")
    plt.show()
else:
    print("Pas d'historique de validation dans ces runs.")
print("Figures et tableau :", os.path.abspath(OUT_DIR))